In [ ]:
import pandas as pd

calendar = pd.read_csv('Data/calendar.csv')
items = pd.read_csv('Data/items_per_season.csv')
customers_fy_16_17 = pd.read_csv('Data/customers_fy_16_17.csv')
customers_fy_23_24 = pd.read_csv('Data/customers_fy_23_24.csv')
items_2016 = pd.read_csv('Data/items_2016.csv')
items_2022 = pd.read_csv('Data/items_2022.csv')
orders_fy_16_17 = pd.read_csv('Data/order_invoice_refunds_fy_16_17.csv')
orders_fy_22_23 = pd.read_csv('Data/order_invoice_refunds_fy_22_23.csv')

In [ ]:
# hier stehen die echten Spaltennamen in Row 0
items = pd.read_csv("Data/items_per_season.csv", skiprows=1, names=["ITEM_NO", "SEASON", "IS_CURRENT_SEASON", "ITEM_END_DATE", "SEASON_START_DATE", "SEASON_END_DATE", "PRODUCT_STATUS_PER_SEASON", "IS_ACTIVE_IN_SEASON"])
items = items.drop(index=0).reset_index(drop=True)
print(items.columns.tolist())
display(items.head())

In [ ]:
orders = pd.concat([orders_fy_16_17, orders_fy_22_23], ignore_index=True)
items_list = pd.concat([items_2016, items_2022], ignore_index=True)


In [ ]:
print(type(items))
print(items.shape)
print(items.columns.tolist())

# Spalten sicher säubern
items.columns = items.columns.astype(str).str.strip()
items_list.columns = items_list.columns.astype(str).str.strip()

# Join-Key säubern
items["ITEM_NO"] = items["ITEM_NO"].astype(str).str.strip()
items_list["ITEM_NO"] = items_list["ITEM_NO"].astype(str).str.strip()

# gewünschte Spalten auswählen
cols = ["ITEM_NO", "SEASON", "ITEM_END_DATE", "SEASON_START_DATE", "SEASON_END_DATE"]

items_add = items.loc[:, cols].copy()

# falls ITEM_NO mehrfach vorkommt:
items_add = items_add.drop_duplicates(subset=["ITEM_NO"])

items_full = items_list.merge(
    items_add,
    how="left",
    on="ITEM_NO"
)

display(items_full.head())

In [ ]:
print("orders:", orders.shape)
items_lookup = items_full[
    ["ITEM_NO", "MAIN_CATEGORY", "SUB_CATEGORY", "PRODUCT_TYPE", "SIZE", "APPEARANCE", "SEASON", "ITEM_END_DATE", "SEASON_START_DATE", "SEASON_END_DATE"]   # hier deine gewünschten Spalten ergänzen
]
print("items_lookup:", items_lookup.shape)

print("Doppelte ITEM_NO in items_lookup:")
print(items_lookup["ITEM_NO"].duplicated().sum())

print(items_lookup["ITEM_NO"].value_counts().head(10))

In [ ]:
agg["APPEARANCE"] = (
    agg["APPEARANCE"]
    .str.strip()        # entfernt Leerzeichen
    .str.lower()        # alles klein
    .str.capitalize()   # erster Buchstabe groß
)
agg["MAIN_CATEGORY"] = (
    agg["MAIN_CATEGORY"]
    .str.strip()        
    .str.lower()        
    .str.capitalize()   
)
agg["SUB_CATEGORY"] = (
    agg["SUB_CATEGORY"]
    .str.strip()        
    .str.lower()        
    .str.capitalize()   
)
agg["PRODUCT_TYPE"] = (
    agg["PRODUCT_TYPE"]
    .str.strip()        
    .str.lower()        
    .str.capitalize()   
)

In [ ]:
items_lookup = items_lookup.drop_duplicates(subset=["ITEM_NO"])

In [ ]:
orders_item = orders.merge(
    items_lookup,
    how="left",
    on="ITEM_NO",
    validate="many_to_one"
)

orders_item = orders_item.sort_values(["ORDER_NO", "LINE_NO"])

display(orders_item.head())

In [ ]:
display(orders_item.tail())

In [ ]:
print("unique order no:", orders_item["ORDER_NO"].nunique())
# echte NaN
print("NaN:", orders_item["ORDER_NO"].isna().sum())

# String "Missing"
print("String Missing:", (orders_item["ORDER_NO"] == "-missing-").sum())

In [ ]:
orders_item["ORDER_NO"] = orders_item["ORDER_NO"].replace("-missing-", pd.NA)
print("NaN:", orders_item["ORDER_NO"].isna().sum())

# String "Missing"
print("String Missing:", (orders_item["ORDER_NO"] == "-missing-").sum())

In [ ]:
orders_item = orders_item.sort_values(["ITEM_NO", "DOCUMENT_NO"])
display(orders_item.head())

In [ ]:
# 1. Check: Gibt es überhaupt „Matching Retouren“?
orders_item.groupby("ITEM_NO")["QUANTITY"].sum().sort_values().head(20)
# wenn 0, dann wurde alles retourniert

In [ ]:
orders_item["ORDER_NO"].value_counts().head(10)

In [ ]:
orders_item[orders_item["ORDER_NO"] == "AU2425-00525541"].sort_values(["DOCUMENT_NO", "ITEM_NO"]).head(20)

In [ ]:
returns = orders_item[orders_item["QUANTITY"] <= 0]
sales = orders_item[orders_item["QUANTITY"] > 0]

In [ ]:
needed_cols = [
    "ORDER_NO", "ITEM_NO", "QUANTITY", "NET_AMOUNT",
    "SEASON", "SEASON_START_DATE", "SEASON_END_DATE",
    "PRODUCT_TYPE", "SIZE", "APPEARANCE", "MAIN_CATEGORY", "SUB_CATEGORY",
    "IS_RETURN_INCL_INDIRECT", "RETURN_REASON_CODE",
    "DAYS_ORDER_TO_INVOICE", "SEASON_ORDER_DATE", "SALES_PRICE_ORIGINAL",
    "SALES_PRICE_INVOICED",
    "UVP"
]

missing_cols = [col for col in needed_cols if col not in orders_item.columns]
print("Fehlende Spalten:", missing_cols)

In [ ]:
orders_item["RETURN_REASON_CODE_CLEAN"] = orders_item["RETURN_REASON_CODE"].fillna("")


agg = orders_item.groupby(["ORDER_NO", "ITEM_NO"], sort=False).agg({
    "QUANTITY": "sum",
    "NET_AMOUNT": "sum",
    "SALES_PRICE_ORIGINAL": "mean",
    "SALES_PRICE_INVOICED": "mean",
    "UVP": "mean",
    "INVOICE_DISCOUNT_AMOUNT": "sum",
    "LINE_DISCOUNT_AMOUNT": "sum",
    "SEASON": "first",
    "SEASON_START_DATE": "first",
    "SEASON_END_DATE": "first",
    "PRODUCT_TYPE": "first",
    "SIZE": "first",
    "APPEARANCE": "first",
    "MAIN_CATEGORY": "first",
    "SUB_CATEGORY": "first",
    "IS_RETURN_INCL_INDIRECT": "max",
    "RETURN_REASON_CODE_CLEAN": "max",
    "DAYS_ORDER_TO_INVOICE": "mean",
    "SEASON_ORDER_DATE": "first"
}).reset_index()

In [ ]:
(agg["QUANTITY"] < 0).sum()
edge_cases = agg[agg["QUANTITY"] < 0]

# Anteil
print("Anteil an allen Bestellungen:", len(edge_cases) / len(agg))

display(edge_cases.head(20))

agg["EDGE_CASE"] = (agg["QUANTITY"] < 0).astype(int) # markiere für später

In [ ]:
agg_clean = agg[agg["QUANTITY"] >= 0]

In [ ]:
# Return Variable
agg["RETURNED"] = (agg["QUANTITY"] <= 0).astype(int)
# Season Variable
agg["SEASON_TYPE"] = agg["SEASON"].str[:2]

In [ ]:
import matplotlib.pyplot as plt

# Return Rate: Sommer vs Winter
season_return = agg.groupby("SEASON_TYPE")["RETURNED"].mean()

season_return.plot(kind="bar")
plt.title("Return Rate: SS vs AW")
plt.ylabel("Return Rate")
plt.show()

In [ ]:
# Most returnes Appearances

appearance_returns = agg.groupby("APPEARANCE")["RETURNED"].mean().sort_values()

appearance_returns.tail(15).plot(kind="barh")
plt.title("Top Return Appearances")
plt.show()

In [ ]:
# Main Cat Returns

agg.groupby("MAIN_CATEGORY")["RETURNED"].mean().sort_values().plot(kind="barh")
plt.title("Return Rate by Main Category")
plt.show()

In [ ]:
# Sub Cat + Season

pivot = agg.pivot_table(
    values="RETURNED",
    index="SUB_CATEGORY",
    columns="SEASON_TYPE",
    aggfunc="mean"
)

pivot.sort_values("AW", ascending=False).head(15).plot(kind="barh")
plt.title("Return Rate by Subcategory & Season")
plt.show()

In [ ]:
# viel gekauft +  viel reture

purchase_vs_return = agg.groupby("MAIN_CATEGORY").agg({
    "QUANTITY": "sum",
    "RETURNED": "mean"
})

print(purchase_vs_return.sort_values("RETURNED", ascending=False))

In [ ]:
footwear = agg[
    agg["MAIN_CATEGORY"].isin(["FOOTWEAR", "FOOTWEAR ACC"])
]
footwear_returns = footwear[
    (footwear["RETURNED"] == 1) &
    (footwear["RETURN_REASON_CODE_CLEAN"].notna()) &
    (footwear["RETURN_REASON_CODE_CLEAN"] != "")
]

reasons = footwear_returns["RETURN_REASON_CODE_CLEAN"].str.split(", ")

reasons_exploded = reasons.explode()
reason_counts = reasons_exploded.value_counts()
print(reason_counts.head(10))

In [ ]:
# Absolut

reason_counts.head(10).plot(kind="barh")
plt.title("Top Return Reasons – Footwear")
plt.xlabel("Count")
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Prozent

reason_share = reason_counts / reason_counts.sum()

reason_share.head(10).plot(kind="barh")
plt.title("Top Return Reasons (%) – Footwear")
plt.xlabel("Share")
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Season Split

footwear_returns["SEASON_TYPE"] = footwear_returns["SEASON"].str[:2]

pivot = footwear_returns.explode("RETURN_REASON_CODE_CLEAN").pivot_table(
    index="RETURN_REASON_CODE_CLEAN",
    columns="SEASON_TYPE",
    aggfunc="size",
    fill_value=0
)

pivot.head(10).plot(kind="barh")
plt.show()

In [ ]:
agg_clean = agg[agg["QUANTITY"] >= 0].copy()

agg_clean["DISCOUNT_RATE"] = 1 - (agg_clean["NET_AMOUNT"] / agg_clean["UVP"]).replace(0, pd.NA)

agg_clean["PRICE_DIFF_TO_UVP"] = agg_clean["UVP"] - agg_clean["NET_AMOUNT"]

agg_clean[["NET_AMOUNT", "UVP", "DISCOUNT_RATE", "PRICE_DIFF_TO_UVP"]].describe()

In [ ]:
agg["DISCOUNT_RATE"] = 1 - (agg["NET_AMOUNT"] / agg["UVP"]).replace(0, pd.NA)

agg["PRICE_DIFF_TO_UVP"] = agg["UVP"] - agg["NET_AMOUNT"]

agg[["NET_AMOUNT", "UVP", "DISCOUNT_RATE", "PRICE_DIFF_TO_UVP"]].describe()

In [ ]:
sales = agg[agg["QUANTITY"] > 0].copy()

In [ ]:
# Which Main Categories are sold most often?
sales_by_category = sales["MAIN_CATEGORY"].value_counts()

sales_by_category.head(10).plot(kind="barh")
plt.title("Top Selling Categories")
plt.xlabel("Number of Sales")
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Which Main Categories are sold most often?

sales["SUB_CATEGORY"].value_counts().head(15).plot(kind="barh")
plt.title("Top Selling Subcategories")
plt.show()

In [ ]:
# Appearance

top_appearance = sales["APPEARANCE"].value_counts().head(15)

top_appearance.plot(kind="barh")
plt.title("Top Selling Appearances")
plt.show()

In [ ]:
# Seasonal Analysis

sales["SEASON_TYPE"] = sales["SEASON"].str[:2]

sales_by_season = sales["SEASON_TYPE"].value_counts()

sales_by_season.plot(kind="bar")
plt.title("Sales by Season (SS vs AW)")
plt.ylabel("Count")
plt.show()

In [ ]:
# Price Analysis

sales["NET_AMOUNT"].hist(bins=50)
plt.title("Distribution of Net Sales")
plt.show()

In [ ]:
agg["NET_AMOUNT"].describe()

In [ ]:
# Discount vs Sales

sales["DISCOUNT_RATE"].hist(bins=50)
plt.title("Discount Distribution (Sold Items)")
plt.show()

In [ ]:
# “Best Products”

best_items = sales.groupby("ITEM_NO").size().sort_values(ascending=False).head(10)

print(best_items)